# **01_train_users_2.csv 전처리**


## 01-01. 환경설정 및 데이터 로드

In [ ]:
# 기본 라이브러리
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 출력 설정
plt.rcParams['font.family'] = 'Malgun Gothic' # 한글 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 데이터 경로
DATA_DIR = "/content/drive/MyDrive/01_project/260226_datathon/"

def load_data():
    train = pd.read_csv(os.path.join(DATA_DIR, "train_users_2.csv"))
    # Test 데이터는 예측용이므로 레이블이 없음을 인지하고 로드
    test = pd.read_csv(os.path.join(DATA_DIR, "test_users.csv"))
    print(f"로드 완료: Train {train.shape}, Test {test.shape}")
    return train, test

df_train, df_test = load_data()

df_train.info()

Mounted at /content/drive
로드 완료: Train (213451, 16), Test (62096, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213451 entries, 0 to 213450
Data columns (total 16 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       213451 non-null  object 
 1   date_account_created     213451 non-null  object 
 2   timestamp_first_active   213451 non-null  int64  
 3   date_first_booking       88908 non-null   object 
 4   gender                   213451 non-null  object 
 5   age                      125461 non-null  float64
 6   signup_method            213451 non-null  object 
 7   signup_flow              213451 non-null  int64  
 8   language                 213451 non-null  object 
 9   affiliate_channel        213451 non-null  object 
 10  affiliate_provider       213451 non-null  object 
 11  first_affiliate_tracked  207386 non-null  object 
 12  signup_app               213451 non-null  ob

## 01-02. 타깃 생성 및 기본 전처리

In [ ]:
# 누수 컬럼 제거 및 타겟 생성

# 결측치 비율 상위 5개(%)
missing_pct = (df_train.isnull().mean() * 100).sort_values(ascending=False)

def create_target_and_drop_leak(df):
    df = df.copy()
    # 타겟: NDF가 아니면 예약 성공(1), 아니면 0
    if 'country_destination' in df.columns:
        df['is_booked'] = (df['country_destination'] != 'NDF').astype(int)

    # 누수 컬럼 제거 (예약자에게만 존재하는 데이터)
    drop_cols = ['date_first_booking']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df

df_train = create_target_and_drop_leak(df_train)
df_test = create_target_and_drop_leak(df_test)


## 01-03. 날짜 및 age 전처리

In [ ]:
def process_dates(df):
    df = df.copy()
    # 날짜 파싱
    df['date_account_created'] = pd.to_datetime(df['date_account_created'])
    df['timestamp_first_active'] = pd.to_datetime(df['timestamp_first_active'].astype(str), format='%Y%m%d%H%M%S')

    # 파생 변수 추출
    df['dac_year'] = df['date_account_created'].dt.year
    df['dac_month'] = df['date_account_created'].dt.month
    df['dac_dayofweek'] = df['date_account_created'].dt.dayofweek
    df['dac_is_weekend'] = df['dac_dayofweek'].isin([5, 6]).astype(int)

    df['tfa_hour'] = df['timestamp_first_active'].dt.hour # 시간 추
    df["tfa_dayofweek"] = df["timestamp_first_active"].dt.dayofweek # 요일 추출

    # 가입과 첫 활동 사이의 차이 (음수 방지: 절대값 처리)
    df['days_lag'] = (df['timestamp_first_active'].dt.normalize() - df['date_account_created']).dt.days.abs()

    return df

df_train = process_dates(df_train)
df_test = process_dates(df_test)

In [ ]:
def clean_age_and_bucket(df):
    df = df.copy()
    # 이상치 결측 처리 (15세 미만, 90세 초과)
    df.loc[(df['age'] < 15) | (df['age'] > 90), 'age'] = np.nan

    # 결측 여부 플래그 생성
    df['age_missing'] = df['age'].isna().astype(int)

    # 5세 단위 구간화 (나중에 Others 처리를 위해 필요)
    def get_bucket(age):
        if pd.isna(age): return "unknown"
        lower = int((age // 5) * 5)
        return f"{lower}-{lower+4}"

    df['age_bucket'] = df['age'].apply(get_bucket)
    return df

df_train = clean_age_and_bucket(df_train)
df_test = clean_age_and_bucket(df_test)

In [ ]:
# age_bucket 및 세션 중복 기기 정보 즉시 삭제
df_train.drop(columns=['age_bucket', 'first_device_type'], errors='ignore', inplace=True)
df_test.drop(columns=['age_bucket', 'first_device_type'], errors='ignore', inplace=True)

## 01-04. 범주형 변수 정리 및 dtype 변환

In [ ]:
cat_cols = ['signup_method', 'signup_flow', 'language', 'affiliate_channel',
            'affiliate_provider', 'first_affiliate_tracked', 'signup_app', 'first_browser', 'gender']

# 범주형 최적화 수행 (1% 미만 Others 통합)
for col in cat_cols:
    # 결측치 및 unknown 처리
    df_train[col] = df_train[col].replace('-unknown-', 'unknown').fillna('unknown')
    df_test[col] = df_test[col].replace('-unknown-', 'unknown').fillna('unknown')

    # Train 기준 1% 미만 범주 찾기
    counts = df_train[col].value_counts(normalize=True)
    rare_categories = counts[counts < 0.01].index.tolist()

    # 'others'로 통합
    df_train[col] = df_train[col].apply(lambda x: 'others' if x in rare_categories else x)
    df_test[col] = df_test[col].apply(lambda x: 'others' if x in rare_categories else x)


## 01-05. 전처리 결과 점검 및 저장

In [ ]:
print("[디버깅] 최종 결측치 비율:")
print(df_train.isnull().mean().sort_values(ascending=False).head(5))
print("\n최종 컬럼 수:", len(df_train.columns))

[디버깅] 최종 결측치 비율:
age                       0.424407
date_account_created      0.000000
id                        0.000000
timestamp_first_active    0.000000
gender                    0.000000
dtype: float64

최종 컬럼 수: 23


In [ ]:
# 변환할 컬럼 리스트 정의 (회의록 기준)
cat_cols = [
    'gender', 'signup_method', 'signup_flow', 'language',
    'affiliate_channel', 'affiliate_provider', 'first_affiliate_tracked',
    'signup_app', 'first_browser', 'country_destination'
]

# 범주형(category) 타입으로 일괄 변환
for col in cat_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype('category')
    if col in df_test.columns:
        df_test[col] = df_test[col].astype('category')

In [ ]:
# 최종 데이터 변수할당
df_train = df_train.copy()

# shape 확인
print(f"최종 데이터 Shape: {df_train.shape}")

최종 데이터 Shape: (213451, 23)


# **02_sessions.csv 전처리**

## 02-01. 데이터 로드 및 기본 설정

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
pd.set_option('display.max_columns', None)

# 파일 불러오기 & 변수할당(코랩용)
from google.colab import drive
drive.mount('/content/drive')

# 파일 불러오기 & 변수할당(코랩용)
base_dir = "/content/drive/MyDrive/01_project/260226_datathon"
sessions_file = os.path.join(base_dir, "sessions.csv")
# train_file = os.path.join(base_dir, "preprocessed_train_users.csv")

df_sessions = pd.read_csv(sessions_file)
# df_train = pd.read_csv(train_file)

print(df_sessions.info())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10567737 entries, 0 to 10567736
Data columns (total 6 columns):
 #   Column         Dtype  
---  ------         -----  
 0   user_id        object 
 1   action         object 
 2   action_type    object 
 3   action_detail  object 
 4   device_type    object 
 5   secs_elapsed   float64
dtypes: float64(1), object(5)
memory usage: 483.8+ MB
None


In [ ]:
cat_cols_sessions = [
    'action',
    'action_type',
    'device_type']

for c in cat_cols_sessions:
    if c in df_sessions.columns:
        df_sessions[c] = df_sessions[c].astype('category')

# action_detail은 경우에 따라 고유값이 너무 많아서 category가 비효율적일 수 있음.
# 일단 object 유지 후 회의 내용에 따라 선별 진행

## 02-02. train 유저 기준 필터링 및 결측치 처리
- 수치형은 파생변수 처리하면서 진행

In [ ]:
# train에 있는 id만 사용하도록 필터링

df_train["id"] = df_train["id"].astype(str)
train_ids = set(df_train["id"].tolist())

df_sessions["user_id"] = df_sessions["user_id"].astype(str)
df_sessions = df_sessions[df_sessions["user_id"].isin(train_ids)].copy()

In [ ]:
# 결측치는 "NaN" 카테고리로 분류
from pandas.api.types import CategoricalDtype
for c in ["action", "action_type", "device_type"]:
    if isinstance(df_sessions[c].dtype, CategoricalDtype):
        if "NaN" not in df_sessions[c].cat.categories:
            df_sessions[c] = df_sessions[c].cat.add_categories(["NaN"])
    else:
        pass

    df_sessions[c] = df_sessions[c].fillna("NaN")# 결측값을 "NaN" 값으로 채우기

In [ ]:
# 이후 파생변수/집계에 사용할 기준 테이블 변수 할당 ----->df_sessions_base
# train 유저 필터링 + 범주형 결측 처리까지 끝난 상태를 기준으로 잡음
df_sessions_base = df_sessions.copy()

## 02-03. 세션 파생변수 생성

### 세션시간

In [ ]:
# df_sessions_base: 전체 이벤트 유지
# df_sessions_time: 시간 집계 전용(secs_elapsed 값이 있는 row만 남긴 버전)
# sess_secs_nonnull: base에 있어서 sess_secs_nonnull_sum 집계 가능

# 전체 이벤트 기준 DF에 시간값 존재 여부 플래그 추가
df_sessions_base["sess_secs_nonnull"] = (
    df_sessions_base["secs_elapsed"].notna().astype("int8"))

# 시간 관련 집계용 df_sessions_time 생성
df_sessions_time = df_sessions_base[
    df_sessions_base["secs_elapsed"].notna()].copy()

print("df_sessions_base",df_sessions_base.shape)
print("df_sessions_time",df_sessions_time.shape)
df_sessions_time

# total 시간과 평균시간은 유저별로 집계할 때 계산

df_sessions_base (5537957, 7)
df_sessions_time (5464142, 7)


,user_id,action,action_type,action_detail,device_type,secs_elapsed,sess_secs_nonnull
0,d1mm9tcy42,lookup,NaN,NaN,Windows Desktop,319.0,1
1,d1mm9tcy42,search_results,click,view_search_results,Windows Desktop,67753.0,1
2,d1mm9tcy42,lookup,NaN,NaN,Windows Desktop,301.0,1
3,d1mm9tcy42,search_results,click,view_search_results,Windows Desktop,22141.0,1
4,d1mm9tcy42,lookup,NaN,NaN,Windows Desktop,435.0,1
...,...,...,...,...,...,...,...
5555275,nw9fwlyb5f,index,data,reservations,iPhone,245.0,1
5555276,nw9fwlyb5f,unavailabilities,data,unavailable_dates,iPhone,286.0,1
5555277,nw9fwlyb5f,notifications,submit,notifications,iPhone,830.0,1
5555278,nw9fwlyb5f,search,click,view_search_results,iPhone,101961.0,1


### 디바이스 분류

In [ ]:
# device_type 문자열 안에 특정 단어가 들어가면 desktop 판별
DESKTOP_PATTERN = r"Desktop"

# user_id x device_type 빈도 테이블
df_device_counts = (
    df_sessions_base[["user_id", "device_type"]]
    .dropna(subset=["device_type"])  # device_type이 없는 행은 제거
    .groupby(["user_id", "device_type"], observed=True)
    .size()
    .rename("device_count")
    .reset_index())

# 사용자별 가장 많이 사용한 device_type(최빈 디바이스)만 남기기
df_device_top = df_device_counts[
    df_device_counts["device_count"]
    == df_device_counts.groupby("user_id")["device_count"].transform("max")].copy()

# 최빈 device_type 후보가 "Desktop" 계열인지 여부를 0/1 플래그로 생성
df_device_top["is_desktop"] = (
    df_device_top["device_type"].astype("string")
    .str.contains(DESKTOP_PATTERN, regex=False, na=False)
    .astype("int8"))

# 사용자별로, 최빈 디바이스 후보들 중 하나라도 Desktop 계열이 있으면 1, 아니면 0
df_user_device = (
    df_device_top.groupby("user_id", as_index=False)["is_desktop"]
    .max()
    .rename(columns={"is_desktop": "main_device_type_d"}))

# 각 사용자가 사용한 device_type의 종류 개수를 계산
df_user_device_count = (
    df_sessions_base
    .groupby("user_id", as_index=False)["device_type"]
    .nunique()
    .rename(columns={"device_type": "device_type_nunique"}))

# -----------------------------------------------------

# 사용자 단위 디바이스 피처 테이블

# 대표 디바이스 플래그(main_device_type_d)와
# 디바이스 종류 수(device_type_nunique)를 하나의 DF로 합치기
df_user_device = df_user_device.merge(
    df_user_device_count, on="user_id", how="outer")

# main_device_type_d 결측은 "0"으로 간주 (Desktop 대표 디바이스가 없다)
df_user_device["main_device_type_d"] = (
    df_user_device["main_device_type_d"].fillna(0).astype("int8"))

# device_type_nunique가 2개 이상이면 여러 디바이스를 쓰는 사용자로 간주해서 1, 아니면 0
df_user_device["multi_device"] = (
    (df_user_device["device_type_nunique"].fillna(0) >= 2).astype("int8"))

# 최종적으로 user_id 단위 피처는 이 3개만 남김
df_user_device = df_user_device[
    ["user_id", "main_device_type_d", "multi_device"]]


# -----------------------------------------------------

# 세션 테이블에 매핑
# df_user_device는 df_sessions_base 기준으로 만들고,
# 이후 df_sessions_base와 df_sessions_time 둘 다에 같은 값으로 매핑

# user_id를 인덱스로 한 매핑 테이블 준비
df_user_device_map = df_user_device.set_index("user_id")

# sessions_base, sessions_time 테이블에 사용자 대표 디바이스 플래그 추가
df_sessions_base["main_device_type_d"] = (
    df_sessions_base["user_id"]
    .map(df_user_device_map["main_device_type_d"])
    .fillna(0) # 매핑 안 되면 0으로 처리
    .astype("int8"))

df_sessions_time["main_device_type_d"] = (
    df_sessions_time["user_id"]
    .map(df_user_device_map["main_device_type_d"])
    .fillna(0)
    .astype("int8"))

# sessions_base, sessions_time 테이블에 multi_device 플래그 매핑
df_sessions_base["multi_device"] = (
    df_sessions_base["user_id"]
    .map(df_user_device_map["multi_device"])
    .fillna(0)
    .astype("int8"))

df_sessions_time["multi_device"] = (
    df_sessions_time["user_id"]
    .map(df_user_device_map["multi_device"])
    .fillna(0)
    .astype("int8"))


# 검증
assert (df_sessions_base.groupby("user_id")["main_device_type_d"].nunique() <= 1).all()
assert (df_sessions_base.groupby("user_id")["multi_device"].nunique() <= 1).all()

### 액션

In [ ]:
# 이벤트 플래그 생성

# 총 액션 횟수 카운트 플래그 -> 각 액션마다 1값을 줘서 user 합계하면 총 액션 갯수 나옴
df_sessions_base["sess_event_cnt_all"] = 1

# action_type별 이벤트 플래그(원핫인코딩으로 처리가능)
df_action_type_flags = pd.get_dummies( df_sessions_base["action_type"],
    prefix="cnt").astype("int8")
# base 테이블에 매핑
df_sessions_base = pd.concat([df_sessions_base, df_action_type_flags],axis=1)


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats.contingency import expected_freq
from statsmodels.stats.multitest import multipletests

train_booked_map = df_train.set_index("id")["is_booked"]
# ---------------------------------------------------------
# 처리방법 요약
# user × action 조합에서 각 action이 예약 여부와 유의한지 카이제곱 검정 수행
# 여러 action을 동시에 검정하므로 FDR(BH) 보정으로 거짓 양성 제어.
# action 들 중 통계적으로 의미 있는(action–예약 간 연관이 유의한) action만 선별후
# 각 유저별로 예약과 가장 강하게 연결된 action 한 개를 대표 action으로 선택
# ---------------------------------------------------------

# sig_action용:
# user 단위 action 여부 테이블
df_user_action = (
    df_sessions_base[["user_id", "action"]]
    .dropna()
    .drop_duplicates())

# booked 매핑
# is_booked가 없는 유저(매핑 실패)는 분석 대상에서 제거
df_user_action["is_booked"] = df_user_action["user_id"].map(train_booked_map)
df_user_action = df_user_action.dropna(subset=["is_booked"])

# 전체 유저 수, 예약한 유저 수, 예약 안 한 유저 수를 계산.
total_users = df_user_action["user_id"].nunique()
total_booked = int(
    df_user_action.drop_duplicates("user_id")["is_booked"].sum())
total_not = int(total_users - total_booked)
if total_booked == 0 or total_not == 0:
    raise ValueError("is_booked가 한 값(0 또는 1)만 존재 → action 유의성 검정 불가")

# df_user_action을 action별로 묶고,
# 각 action 수행여부(0/1) 대해 예약여부(0/1)를 결합해 2×2 넘파이 표생성
results = []
for action, group in df_user_action.groupby("action", observed=True):
    booked_yes = int(group["is_booked"].sum())
    booked_no = int(len(group) - booked_yes)
    not_action_yes = int(total_booked - booked_yes)
    not_action_no = int(total_not - booked_no)
    table = np.array(
        [[booked_yes, booked_no],
         [not_action_yes, not_action_no]],
        dtype=int)
    if (table < 0).any(): #
        continue
    if (table.sum(axis=0) == 0).any() or (table.sum(axis=1) == 0).any():
        continue
    exp = expected_freq(table)
    if (exp == 0).any():
        continue
    chi2, p, _, _ = chi2_contingency(table, correction=False)
    results.append((action, p, len(group)))

# action별 검정 결과를 한 데이터프레임으로 정리
df_action_summary = pd.DataFrame(
    results,
    columns=["action", "p_value", "n_users"])
if df_action_summary.empty:
    raise ValueError("유효한 2x2 분할표가 없어 검정을 수행할 action이 없습니다.")

# chi2_contingency : 위 2×2 표에 대해 카이제곱 검정을 수행하고 p값 계산
reject, p_adj, _, _ = multipletests(
    df_action_summary["p_value"],
    method="fdr_bh")

# ---------------------------------------------------------------------------
# 다중 검정 보정(FDR, Benjamini–Hochberg)
# action 종류가 많아지면, 관련이 없어도 우연히 p<0.05가 많이 생길 수 있음
# Benjamini–Hochberg 방법으로 p값을 조정해서,
# 우연히 나온 작은 p값을 줄이고, 실제 의미있는 action만 남기려는 목적

# 예약 여부와 통계적으로 유의한 연관이 있는 action들을 모아서
# p_adj가 작은 순으로 정렬해 가장 강한 것부터 검토
df_action_summary["p_adj"] = p_adj
df_action_summary["significant"] = reject
df_significant_actions = df_action_summary[
    (df_action_summary["significant"]) &
    (df_action_summary["n_users"] >= 0)].sort_values("p_adj")

# 유의한 action에 대해서만, 맵핑 테이블로 사용
df_sig_action_map = df_significant_actions[
    ["action", "p_adj", "n_users"]].copy()

# 유저마다 가장 강한 sig action 하나 선택
df_sig_action_users = df_user_action.merge(
    df_sig_action_map,
    on="action",
    how="inner")

# 각 유저에게 가장 유의하게 예약과 연결된 action 한 개를 대표 action으로 선택
df_user_top_action = (
    df_sig_action_users
    .sort_values(
        ["user_id", "p_adj", "n_users", "action"],
        ascending=[True, True, False, True])
    .drop_duplicates(subset=["user_id"], keep="first")
    .reset_index(drop=True))

# 모든 유저에 대해, 위에서 뽑은 대표 sig action 정보를 left join
# sig action이 없는 유저는 해당 컬럼이 NaN
df_all_action_users = df_user_action[["user_id"]].drop_duplicates().copy()
df_user_top_action_all = (
    df_all_action_users
    .merge(df_user_top_action, on="user_id", how="left")
    .sort_values("user_id")
    .reset_index(drop=True))

df_significant_actions, df_user_top_action_all


(                       action   p_value  n_users     p_adj  significant
 34      ajax_refresh_subtotal  0.000000    34523  0.000000         True
 23          ajax_image_upload  0.000000     5166  0.000000         True
 59      cancellation_policies  0.000000     5789  0.000000         True
 90                  dashboard  0.000000    29515  0.000000         True
 74            complete_status  0.000000     3904  0.000000         True
 ..                        ...       ...      ...       ...          ...
 66                      check  0.015193       43  0.028023         True
 193  payoneer_signup_complete  0.015319       18  0.028098         True
 35          ajax_send_message  0.017025       20  0.030887         True
 170             multi_message  0.017025       20  0.030887         True
 210               photography  0.021455      142  0.038712         True
 
 [184 rows x 5 columns],
           user_id          action  is_booked          p_adj  n_users
 0      00023iyk9l  header_

In [ ]:
# ---------------------------------------------------------
# 처리방법 이나 설명은 sig_action용 코드 설명과 동일함
# ---------------------------------------------------------

# sig_action_detail용:
# user 단위 action_detail 여부 테이블
df_user_action_detail = (
    df_sessions_base[["user_id", "action_detail"]]
    .dropna()
    .drop_duplicates())

# booked 매핑
df_user_action_detail["is_booked"] = (
    df_user_action_detail["user_id"].map(train_booked_map))

df_user_action_detail = df_user_action_detail.dropna(subset=["is_booked"])
total_users = df_user_action_detail["user_id"].nunique()
total_booked = int(
    df_user_action_detail.drop_duplicates("user_id")["is_booked"].sum())

total_not = int(total_users - total_booked)
if total_booked == 0 or total_not == 0:
    raise ValueError("is_booked가 한 값(0 또는 1)만 존재 → action_detail 유의성 검정 불가")
results = []
for action_detail, group in df_user_action_detail.groupby("action_detail", observed=True):
    booked_yes = int(group["is_booked"].sum())
    booked_no = int(len(group) - booked_yes)
    not_action_yes = int(total_booked - booked_yes)
    not_action_no = int(total_not - booked_no)
    table = np.array(
        [[booked_yes, booked_no],
         [not_action_yes, not_action_no]],
        dtype=int)

    if (table < 0).any():
        continue
    if (table.sum(axis=0) == 0).any() or (table.sum(axis=1) == 0).any():
        continue
    exp = expected_freq(table)
    if (exp == 0).any():
        continue
    chi2, p, _, _ = chi2_contingency(table, correction=False)
    results.append((action_detail, p, len(group)))

df_action_detail_summary = pd.DataFrame(
    results,
    columns=["action_detail", "p_value", "n_users"])

if df_action_detail_summary.empty:
    raise ValueError("유효한 2x2 분할표가 없어 검정을 수행할 action_detail이 없습니다.")
reject, p_adj, _, _ = multipletests(
    df_action_detail_summary["p_value"],
    method="fdr_bh")

df_action_detail_summary["p_adj"] = p_adj
df_action_detail_summary["significant"] = reject
df_significant_actions_detail = df_action_detail_summary[
    (df_action_detail_summary["significant"]) &
    (df_action_detail_summary["n_users"] >= 0)
].sort_values("p_adj")

df_sig_action_detail_map = df_significant_actions_detail[
    ["action_detail", "p_adj", "n_users"]
].copy()

df_sig_action_detail_users = df_user_action_detail.merge(
    df_sig_action_detail_map,
    on="action_detail",
    how="inner")

df_user_top_action_detail = (
    df_sig_action_detail_users
    .sort_values(
        ["user_id", "p_adj", "n_users", "action_detail"],
        ascending=[True, True, False, True])
    .drop_duplicates(subset=["user_id"], keep="first")
    .reset_index(drop=True))

df_all_action_detail_users = (
    df_user_action_detail[["user_id"]].drop_duplicates().copy())

df_user_top_action_detail_all = (
    df_all_action_detail_users
    .merge(df_user_top_action_detail, on="user_id", how="left")
    .sort_values("user_id")
    .reset_index(drop=True))

df_significant_actions_detail, df_user_top_action_detail_all


(                  action_detail   p_value  n_users     p_adj  significant
 17        cancellation_policies  0.000000     5789  0.000000         True
 13                at_checkpoint  0.000000     2549  0.000000         True
 26           confirm_email_link  0.000000    30857  0.000000         True
 23  change_trip_characteristics  0.000000    34523  0.000000         True
 32         create_phone_numbers  0.000000    13814  0.000000         True
 ..                          ...       ...      ...       ...          ...
 96                 signup_modal  0.007405      603  0.009771         True
 80            popular_wishlists  0.008106      673  0.010587         True
 19          change_availability  0.010275       15  0.013285         True
 14                      book_it  0.015426     1773  0.019746         True
 59                   login_page  0.023243     1780  0.029457         True
 
 [101 rows x 5 columns],
           user_id   action_detail  is_booked          p_adj  n_users
 0 

In [ ]:
# user_id 유일성 검증 (디버깅)
assert df_user_top_action_all["user_id"].is_unique
assert df_user_top_action_detail_all["user_id"].is_unique

In [ ]:
# sig_action-base 매핑

# 대표값으로 선정한 action, action_detail을 각각 sig_action, sig_action_detail로 매핑

# action -> sig_action
df_sessions_base["sig_action"] = df_sessions_base["user_id"].map(
    df_user_top_action_all.set_index("user_id")["action"])

# action_detail -> sig_action_detail
df_sessions_base["sig_action_detail"] = df_sessions_base["user_id"].map(
    df_user_top_action_detail_all.set_index("user_id")["action_detail"])

df_sessions_base.head()
# sig_action, sig_action_detail 카운팅은 유저 id별 집계할 때 수행

,user_id,action,action_type,action_detail,device_type,secs_elapsed,sess_secs_nonnull,main_device_type_d,multi_device,sess_event_cnt_all,cnt_-unknown-,cnt_booking_request,cnt_booking_response,cnt_click,cnt_data,cnt_message_post,cnt_modify,cnt_partner_callback,cnt_submit,cnt_view,cnt_NaN,sig_action,sig_action_detail
0,d1mm9tcy42,lookup,NaN,NaN,Windows Desktop,319.0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,header_userpic,header_userpic
1,d1mm9tcy42,search_results,click,view_search_results,Windows Desktop,67753.0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,header_userpic,header_userpic
2,d1mm9tcy42,lookup,NaN,NaN,Windows Desktop,301.0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,header_userpic,header_userpic
3,d1mm9tcy42,search_results,click,view_search_results,Windows Desktop,22141.0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,header_userpic,header_userpic
4,d1mm9tcy42,lookup,NaN,NaN,Windows Desktop,435.0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,header_userpic,header_userpic


## 02-04. user_id 기준 집계 및 검증

In [ ]:
#  집계 대상 컬럼 설정
CNT_COLS = [c for c in df_sessions_base.columns if c.startswith("cnt_")]


#  user_id 기준  기본집계 테이블 생성
# 기준 테이블: df_sessions_base
# - cnt_* : 전체 행동 로그 수
# - sess_event_cnt_sum : 전체 이벤트 수
# - sess_secs_nonnull_sum : 시간값이 존재하는 이벤트 수
# - main_device_type_d / multi_device : 메인 디바이스가 데스크탑인지 여부
# - sig_action / sig_action_detail : 유저별 유의미한 action, sig_action

df_user_base = (
    df_sessions_base
    .groupby("user_id", as_index=False)
    .agg(
        **{c: (c, "sum") for c in CNT_COLS},
        sess_event_cnt_sum=("sess_event_cnt_all", "sum"),
        sess_secs_nonnull_sum=("sess_secs_nonnull", "sum"),
        main_device_type_d=("main_device_type_d", "first"),
        multi_device=("multi_device", "first"),
        sig_action=("sig_action", "first"),
        sig_action_detail=("sig_action_detail", "first"),))

# 시간 관련 집계 테이블 생성
# 기준 테이블: df_sessions_time
# - secs_elapsed가 실제로 존재하는 row만 대상으로 집계(df_sessions_time 사용)
df_user_time = (df_sessions_time
    .groupby("user_id", as_index=False)
    .agg(
        # 시간 총합
        secs_elapsed_sum=("secs_elapsed", "sum"),
        # 시간 중앙값
        secs_median=("secs_elapsed", "median"),))

# 기본 집계(base) + 시간 집계(time) 결합
df_user_base = df_user_base.merge(
    df_user_time,
    on="user_id",
    how="left")

# 평균 체류시간 계산
# 기준:
# - 분자: secs_elapsed_sum (time 기준 합계)
# - 분모: sess_secs_nonnull_sum (시간값 있는 이벤트 수)
df_user_base["secs_avr"] = (
    df_user_base["secs_elapsed_sum"] / df_user_base["sess_secs_nonnull_sum"]
).replace([np.inf, -np.inf], np.nan)

df_user_base[
    ["secs_elapsed_sum", "sess_secs_nonnull_sum", "sess_event_cnt_sum", "secs_avr"]
].head()


# user_id별 대표 sig_action 발생 횟수 계산
df_user_action_count = (
    df_sessions_base
    .groupby(["user_id", "action"], observed=True)
    .size()
    .rename("sig_action_cnt")
    .reset_index())

df_user_features = df_user_base.merge(
    df_user_action_count,
    left_on=["user_id", "sig_action"],
    right_on=["user_id", "action"],
    how="left",)

df_user_features.drop(columns=["action"], inplace=True)
df_user_features["sig_action_cnt"] = (
    df_user_features["sig_action_cnt"].fillna(0).astype("int64"))


# user_id별 sig_action_detail 발생 횟수 계산
df_user_action_detail_count = (
    df_sessions_base
    .groupby(["user_id", "action_detail"], observed=True)
    .size()
    .rename("sig_action_detail_cnt")
    .reset_index())

df_user_features = df_user_features.merge(
    df_user_action_detail_count,
    left_on=["user_id", "sig_action_detail"],
    right_on=["user_id", "action_detail"],
    how="left",)

df_user_features.drop(columns=["action_detail"], inplace=True)

df_user_features["sig_action_detail_cnt"] = (
    df_user_features["sig_action_detail_cnt"].fillna(0).astype("int64"))

In [ ]:
# 집계 최종 검증

# user_id 1행 보장
assert df_user_features["user_id"].is_unique

# time 테이블 user는 항상 base 테이블 user의 부분집합이어야 함
assert set(df_sessions_time["user_id"]).issubset(set(df_sessions_base["user_id"]))

# 디바이스 관련 컬럼이 user_id 내에서 실제로 단일값인지 점검
assert (df_sessions_base.groupby("user_id")["main_device_type_d"].nunique() <= 1).all()
assert (df_sessions_base.groupby("user_id")["multi_device"].nunique() <= 1).all()

# 시간값 있는 이벤트 수는 전체 이벤트 수보다 클 수 없음
assert (
    df_user_features["sess_secs_nonnull_sum"].fillna(0)
    <= df_user_features["sess_event_cnt_sum"]
).all()

# merge 직전에 사용할 세션 피처 테이블 복사본
df_sessions_features = df_user_features.copy(deep=True)

# **03_train-session merge 및 데이터 정리**

In [ ]:
# 키 컬럼 존재 확인
assert "user_id" in df_sessions_features.columns
assert "id" in df_train.columns

# df_train.id <-> df_sessions_features.user_id 로 조인
df_merged = df_train.merge(
    df_sessions_features,
    left_on="id",
    right_on="user_id",
    how="left")

# 조인 후 키 컬럼 정리
df_merged.drop(columns=["user_id"], inplace=True)
df_merged.rename(columns={"id": "user_id"}, inplace=True)
df_merged.head()

,user_id,date_account_created,timestamp_first_active,gender,age,signup_method,signup_flow,language,affiliate_channel,affiliate_provider,first_affiliate_tracked,signup_app,first_browser,country_destination,is_booked,dac_year,dac_month,dac_dayofweek,dac_is_weekend,tfa_hour,tfa_dayofweek,days_lag,age_missing,cnt_-unknown-,cnt_booking_request,cnt_booking_response,cnt_click,cnt_data,cnt_message_post,cnt_modify,cnt_partner_callback,cnt_submit,cnt_view,cnt_NaN,sess_event_cnt_sum,sess_secs_nonnull_sum,main_device_type_d,multi_device,sig_action,sig_action_detail,secs_elapsed_sum,secs_median,secs_avr,sig_action_cnt,sig_action_detail_cnt
0,gxn3p5htnn,2010-06-28,2009-03-19 04:32:55,unknown,NaN,facebook,0,en,direct,direct,untracked,Web,Chrome,NDF,0,2010,6,0,0,4,3,466,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,820tgsjxq7,2011-05-25,2009-05-23 17:48:09,MALE,38.0,facebook,0,en,seo,google,untracked,Web,Chrome,NDF,0,2011,5,2,0,17,5,732,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4ft3gnwmtx,2010-09-28,2009-06-09 23:12:47,FEMALE,56.0,basic,3,en,direct,direct,untracked,Web,IE,US,1,2010,9,1,0,23,1,476,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bjjt8pjhuk,2011-12-05,2009-10-31 06:01:29,FEMALE,42.0,facebook,0,en,direct,direct,untracked,Web,Firefox,other,1,2011,12,0,0,6,5,765,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,87mebub9p4,2010-09-14,2009-12-08 06:11:05,unknown,41.0,basic,0,en,direct,direct,untracked,Web,Chrome,US,1,2010,9,1,0,6,1,280,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# merge 후 shape 확인
print(df_train.shape)
print(df_merged.shape)

(213451, 23)
(213451, 45)


In [ ]:
#  세션 존재 여부 플래그 생성
df_merged["has_session"] = df_merged["sess_event_cnt_sum"].notna().astype("int8")


# 세션 관련 결측 채우기
# - cnt_*, sess_*, secs_*, *_cnt -> 0
# - main_device_type_d, multi_device -> 0
# - sig_action, sig_action_detail은 상태를 구분해서 채움

session_num_cols = [
    c for c in df_merged.columns
    if c.startswith("cnt_")
    or c.startswith("sess_")
    or c.startswith("secs_")
    or c in ["sig_action_cnt", "sig_action_detail_cnt"]
]
session_binary_cols = [
    c for c in ["main_device_type_d", "multi_device"]
    if c in df_merged.columns
]
df_merged[session_num_cols] = df_merged[session_num_cols].fillna(0)
if session_binary_cols:
    df_merged[session_binary_cols] = df_merged[session_binary_cols].fillna(0)


# 세션 자체가 없고 sig_action이 비어 있으면 → "NO_SESSION"
# 세션은 있는데 sig_action, sig_action_detail이 비어 있으면 → "NO_SIG_ACTION","NO_SIG_ACTION_DETAIL"
if "sig_action" in df_merged.columns:
    df_merged.loc[
        (df_merged["has_session"] == 0) & (df_merged["sig_action"].isna()),
        "sig_action"
    ] = "NO_SESSION"
    df_merged.loc[
        (df_merged["has_session"] == 1) & (df_merged["sig_action"].isna()),
        "sig_action"
    ] = "NO_SIG_ACTION"

if "sig_action_detail" in df_merged.columns:
    df_merged.loc[
        (df_merged["has_session"] == 0) & (df_merged["sig_action_detail"].isna()),
        "sig_action_detail"
    ] = "NO_SESSION"
    df_merged.loc[
        (df_merged["has_session"] == 1) & (df_merged["sig_action_detail"].isna()),
        "sig_action_detail"
    ] = "NO_SIG_ACTION_DETAIL"


# dtype 정리
for c in session_num_cols:
    if c.startswith("cnt_") or c.startswith("sess_") or c.endswith("_cnt"):
        df_merged[c] = df_merged[c].astype("int64")
    else:
        df_merged[c] = df_merged[c].astype("float64")

for c in session_binary_cols:
    df_merged[c] = df_merged[c].astype("int8")


# 원본 날짜 문자열 정리
drop_date_cols = ["date_account_created", "timestamp_first_active"]
df_merged.drop(columns=drop_date_cols, inplace=True, errors="ignore")

In [ ]:
# 범주형 컬럼 문자열 정리
base_cat_cols = [
    "gender",
    "signup_method",
    "language",
    "affiliate_channel",
    "affiliate_provider",
    "first_affiliate_tracked",
    "signup_app",
    "first_browser",  ]
for col in base_cat_cols:
    if col in df_merged.columns:
        s = df_merged[col].astype("string").str.strip()
        s = s.replace({
            "": pd.NA,
            "None": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
        })
        df_merged[col] = s.fillna("unknown")

# sig_action 계열은 NO_SESSION / NO_SIG 의미를 보존하면서 정리
if "sig_action" in df_merged.columns:
    s = df_merged["sig_action"].astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "None": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
    })
    if "has_session" in df_merged.columns:
        s = s.mask(s.isna() & (df_merged["has_session"] == 0), "NO_SESSION")
        s = s.mask(s.isna() & (df_merged["has_session"] == 1), "NO_SIG_ACTION")
    else:
        s = s.fillna("NO_SESSION")
    df_merged["sig_action"] = s
if "sig_action_detail" in df_merged.columns:
    s = df_merged["sig_action_detail"].astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "None": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
    })
    if "has_session" in df_merged.columns:
        s = s.mask(s.isna() & (df_merged["has_session"] == 0), "NO_SESSION")
        s = s.mask(s.isna() & (df_merged["has_session"] == 1), "NO_SIG_ACTION_DETAIL")
    else:
        s = s.fillna("NO_SESSION")
    df_merged["sig_action_detail"] = s

# 타깃은 공백만 정리하고 값 자체는 보존
if "country_destination" in df_merged.columns:
    df_merged["country_destination"] = (
        df_merged["country_destination"]
        .astype("string")
        .str.strip()      )

In [ ]:
# 파일 로컬 다운로드  (필요시 주석해제)
# df_merged.to_csv(
#     "preprocessed_merged_reviewed_0302.csv",
#     index=False,
#     encoding="utf-8-sig")
#
# # 구글 코랩용
# from google.colab import files
# files.download("preprocessed_merged_reviewed_0302.csv")
